# HH API captcha probe

This notebook isolates `GET /vacancies` probing from the main crawler.

Goals:
- measure how quickly vacancy search starts returning `403`
- capture raw evidence for the transition from the last `200` to the first `403`
- compare different request pacing and area rotation strategies without touching the main application flow

Known facts from the 2026-03-22 soak run:
- the first blocked vacancy-search request appeared after 119 successful vacancy-search requests
- the first blocked request in the full run used `area=1003`, `page=3`, `per_page=20`
- the raw error body contained `errors[0].type == "captcha_required"`

Official docs worth keeping open while running experiments:
- README: https://github.com/hhru/api/blob/master/README.md
- Errors: https://github.com/hhru/api/blob/master/docs/errors.md
- Vacancy search Redoc: https://api.hh.ru/openapi/redoc#tag/Poisk-vakansij/operation/get-vacancies
- Public OpenAPI spec: https://api.hh.ru/openapi/specification/public

What the official docs say:
- `docs/errors.md` documents `captcha_required` as a normal API error class for some protected operations.
- the public OpenAPI spec for `GET /vacancies` documents `403` as a captcha response.
- the public OpenAPI spec requires the `HH-User-Agent` header.

Operational note:
- the main app has already been getting valid `200` responses with the standard `User-Agent` header.
- to remove ambiguity in this notebook, requests send both `User-Agent` and `HH-User-Agent` with the same value.

This notebook intentionally avoids importing the main project package. Instead it uses a small local helper module in `notebooks/` so experiments stay isolated, reproducible, and easier to extend.
- Each probe run now writes a companion `*-report.json` next to the raw `.jsonl` records.


In [ ]:
import json

from notebooks.hh_api_probe_harness import (
    DEFAULT_SOAK_RUN_ID,
    burst_replay_request_sequence,
    hh_get,
    load_jsonl,
    load_soak_search_sequence,
    make_probe_report_path,
    notebook_config,
    print_summary,
    replay_request_sequence,
    run_fixed_request_probe,
    run_round_robin_probe,
    run_sequential_page_probe,
)

print(notebook_config())


In [ ]:
# Probe helpers now live in notebooks.hh_api_probe_harness.
# Keep the notebook focused on scenarios and interpretation rather than helper internals.


In [ ]:
# Request runners, timing metadata, and report writers are imported above.
# Every saved probe run now gets a sibling `*-report.json` artifact.


In [ ]:
# print_summary(records, records_path=path) now emits a normalized summary.
# Compare sibling `*-report.json` files instead of hand-reading raw JSONL.


In [5]:
# Suggested experiments

# 1. Single request smoke test using the known area from the soak transition.
# smoke = hh_get("/vacancies", {"area": "1003", "page": 0, "per_page": 20})
# print(json.dumps({
#     "status_code": smoke["status_code"],
#     "error_type": smoke["error_type"],
#     "request_id": smoke["request_id"],
#     "captcha_url_with_backurl": smoke["captcha_url_with_backurl"],
# }, indent=2, ensure_ascii=False))

# 2. Repeat the exact same request a few times.
# records, path = run_fixed_request_probe(
#     params={"area": "1003", "page": 0, "per_page": 20},
#     repeats=8,
#     sleep_seconds=5.0,
#     label="fixed-area-1003-page-0",
# )
# print(path)
# print(make_probe_report_path(path))
# print_summary(records, records_path=path)

# 3. Gentle sequential probe against one area.
# records, path = run_sequential_page_probe(
#     area="1003",
#     start_page=0,
#     pages=10,
#     per_page=20,
#     sleep_seconds=2.0,
# )
# print(path)
# print(make_probe_report_path(path))
# print_summary(records, records_path=path)

# 4. Round-robin probe across a few areas to imitate crawler fan-out.
# records, path = run_round_robin_probe(
#     areas=["1", "2", "3", "1003"],
#     pages_per_area=3,
#     per_page=20,
#     sleep_seconds=1.0,
# )
# print(path)
# print(make_probe_report_path(path))
# print_summary(records, records_path=path)

# 5. Load the real soak prefix from Postgres and inspect it.
# source_records = load_soak_search_sequence(
#     crawl_run_id=DEFAULT_SOAK_RUN_ID,
#     limit=130,
# )
# print_summary(source_records)

# 6. Replay the real soak prefix in the original request order.
# source_records = load_soak_search_sequence(
#     crawl_run_id=DEFAULT_SOAK_RUN_ID,
#     limit=130,
# )
# records, path = replay_request_sequence(
#     source_records,
#     header_mode="app_like",
#     sleep_seconds=0.25,
#     label="replay-soak-prefix-130",
# )
# print(path)
# print(make_probe_report_path(path))
# print_summary(records, records_path=path)

# 7. Burst replay the same historical prefix with app-like headers.
# source_records = load_soak_search_sequence(
#     crawl_run_id=DEFAULT_SOAK_RUN_ID,
#     limit=130,
# )
# records, path = burst_replay_request_sequence(
#     source_records,
#     workers=4,
#     header_mode="app_like",
#     label="burst-replay-soak-prefix-130-workers-4",
# )
# print(path)
# print(make_probe_report_path(path))
# print_summary(records, records_path=path)

# 8. Re-load a saved run later and re-check the transition summary.
# saved = load_jsonl(path)
# print(make_probe_report_path(path))
# print_summary(saved, records_path=path)


## How to interpret results

- If the transition is abrupt (`200` -> `403 captcha_required` and then all later requests stay `403`), that looks like a stateful anti-bot block rather than a malformed single request.
- If only some areas fail while others keep returning `200`, then the block may be query-shape specific instead of global.
- If the docs-provided captcha URL works in the browser and the same request succeeds afterwards, that confirms the challenge is operating exactly as documented.
- Keep one variable fixed at a time: network path, user-agent, sleep between requests, area rotation pattern, and request volume.
- Save each probe run and compare `last_success` vs `first_403` before drawing conclusions from any one run.
- Compare the saved `*-report.json` files by `scenario_label`, `scenario_type`, `endpoint_kind`, `auth_mode`, `header_mode`, `workers`, `pause_seconds`, `seconds_since_previous_request`, and `minutes_since_first_captcha`.


In [6]:
# 1. Single request smoke test using the known area from the soak transition.
smoke = hh_get("/vacancies", {"area": "1003", "page": 0, "per_page": 20})
print(json.dumps({
    "status_code": smoke["status_code"],
    "error_type": smoke["error_type"],
    "request_id": smoke["request_id"],
    "captcha_url_with_backurl": smoke["captcha_url_with_backurl"],
}, indent=2, ensure_ascii=False))

{
  "status_code": 200,
  "error_type": null,
  "request_id": null,
  "captcha_url_with_backurl": null
}


In [7]:
# 2. Repeat the exact same request a few times.
records, path = run_fixed_request_probe(
    params={"area": "1003", "page": 0, "per_page": 20},
    repeats=8,
    sleep_seconds=5.0,
    label="fixed-area-1003-page-0",
)
print(path)
print(make_probe_report_path(path))
print_summary(records, records_path=path)

{'attempt': 1, 'status_code': 200, 'latency_ms': 224.25, 'items_count': 20, 'error_type': None}
{'attempt': 2, 'status_code': 200, 'latency_ms': 683.22, 'items_count': 20, 'error_type': None}
{'attempt': 3, 'status_code': 200, 'latency_ms': 250.83, 'items_count': 20, 'error_type': None}
{'attempt': 4, 'status_code': 200, 'latency_ms': 217.38, 'items_count': 20, 'error_type': None}
{'attempt': 5, 'status_code': 200, 'latency_ms': 267.44, 'items_count': 20, 'error_type': None}
{'attempt': 6, 'status_code': 200, 'latency_ms': 250.96, 'items_count': 20, 'error_type': None}
{'attempt': 7, 'status_code': 200, 'latency_ms': 441.5, 'items_count': 20, 'error_type': None}
{'attempt': 8, 'status_code': 200, 'latency_ms': 345.49, 'items_count': 20, 'error_type': None}
/home/yurizinyakov/projects/hh_collector/.state/reports/hh-api-probe/20260322T201641Z-07ea92b5-fixed-area-1003-page-0.jsonl
{
  "total_requests": 8,
  "status_counts": {
    "200": 8
  },
  "first_403_index": null,
  "first_captcha_i

In [8]:
# 3. Gentle sequential probe against one area.
records, path = run_sequential_page_probe(
    area="1003",
    start_page=0,
    pages=10,
    per_page=20,
    sleep_seconds=2.0,
)
print(path)
print(make_probe_report_path(path))
print_summary(records, records_path=path)

{'page': 0, 'status_code': 200, 'latency_ms': 223.97, 'items_count': 20, 'error_type': None}
{'page': 1, 'status_code': 200, 'latency_ms': 340.41, 'items_count': 20, 'error_type': None}
{'page': 2, 'status_code': 200, 'latency_ms': 250.87, 'items_count': 20, 'error_type': None}
{'page': 3, 'status_code': 200, 'latency_ms': 243.54, 'items_count': 20, 'error_type': None}
{'page': 4, 'status_code': 200, 'latency_ms': 272.93, 'items_count': 20, 'error_type': None}
{'page': 5, 'status_code': 200, 'latency_ms': 268.23, 'items_count': 20, 'error_type': None}
{'page': 6, 'status_code': 200, 'latency_ms': 298.43, 'items_count': 20, 'error_type': None}
{'page': 7, 'status_code': 200, 'latency_ms': 312.52, 'items_count': 20, 'error_type': None}
{'page': 8, 'status_code': 200, 'latency_ms': 253.01, 'items_count': 20, 'error_type': None}
{'page': 9, 'status_code': 200, 'latency_ms': 277.89, 'items_count': 20, 'error_type': None}
/home/yurizinyakov/projects/hh_collector/.state/reports/hh-api-probe/2

In [9]:
# 4. Round-robin probe across a few areas to imitate crawler fan-out.
records, path = run_round_robin_probe(
    areas=["1", "2", "3", "1003"],
    pages_per_area=3,
    per_page=20,
    sleep_seconds=1.0,
)
print(path)
print(make_probe_report_path(path))
print_summary(records, records_path=path)

{'area': '1', 'page': 0, 'status_code': 200, 'latency_ms': 357.85, 'error_type': None}
{'area': '2', 'page': 0, 'status_code': 200, 'latency_ms': 274.58, 'error_type': None}
{'area': '3', 'page': 0, 'status_code': 200, 'latency_ms': 764.84, 'error_type': None}
{'area': '1003', 'page': 0, 'status_code': 200, 'latency_ms': 268.57, 'error_type': None}
{'area': '1', 'page': 1, 'status_code': 200, 'latency_ms': 319.5, 'error_type': None}
{'area': '2', 'page': 1, 'status_code': 200, 'latency_ms': 272.23, 'error_type': None}
{'area': '3', 'page': 1, 'status_code': 200, 'latency_ms': 501.81, 'error_type': None}
{'area': '1003', 'page': 1, 'status_code': 200, 'latency_ms': 1262.3, 'error_type': None}
{'area': '1', 'page': 2, 'status_code': 200, 'latency_ms': 284.75, 'error_type': None}
{'area': '2', 'page': 2, 'status_code': 200, 'latency_ms': 298.39, 'error_type': None}
{'area': '3', 'page': 2, 'status_code': 200, 'latency_ms': 334.02, 'error_type': None}
{'area': '1003', 'page': 2, 'status_co

In [11]:
# 5. Load the real soak prefix from Postgres and inspect it.
source_records = load_soak_search_sequence(
    crawl_run_id=DEFAULT_SOAK_RUN_ID,
    limit=130,
)
print_summary(source_records)

{
  "total_requests": 130,
  "status_counts": {
    "200": 119,
    "403": 11
  },
  "first_403_index": 119,
  "first_captcha_index": null,
  "ok_before_first_403": 119,
  "ok_after_first_403": 0,
  "forbidden_after_first_403": 10,
  "first_captcha_request_id": null,
  "first_captcha_url": null,
  "first_captcha_url_with_backurl": null
}
{
  "last_success": {
    "timestamp_utc": "2026-03-22T00:42:37.580091+00:00",
    "params": {
      "area": "1003",
      "page": 2,
      "per_page": 20
    },
    "status_code": 200,
    "latency_ms": 232,
    "found": 1237,
    "pages": 62,
    "items_count": 20,
    "request_id": null
  },
  "first_403": {
    "timestamp_utc": "2026-03-22T00:42:37.859774+00:00",
    "params": {
      "area": "1003",
      "page": 3,
      "per_page": 20
    },
    "status_code": 403,
    "latency_ms": 213,
    "error_type": null,
    "error_value": "captcha_required",
    "captcha_url": "https://hh.ru/account/captcha?state=pxvcxBozfu7ry7R4QCetFqDzUL2cbK2LGYV-JLOJ6

In [12]:
# 6. Replay the real soak prefix in the original request order.
source_records = load_soak_search_sequence(
    crawl_run_id=DEFAULT_SOAK_RUN_ID,
    limit=130,
)
records, path = replay_request_sequence(
    source_records,
    header_mode="app_like",
    sleep_seconds=0.25,
    label="replay-soak-prefix-130",
)
print(path)
print(make_probe_report_path(path))
print_summary(records, records_path=path)

{'replay_index': 1, 'source_request_log_id': 170, 'page': 0, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 2, 'source_request_log_id': 171, 'page': 1, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 3, 'source_request_log_id': 172, 'page': 2, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 4, 'source_request_log_id': 173, 'page': 3, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 5, 'source_request_log_id': 174, 'page': 4, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 6, 'source_request_log_id': 175, 'page': 5, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 7, 'source_request_log_id': 176, 'page': 6, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 8, 'source_request_log_id': 177, 'page': 7, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 9, 'source_request_log_id': 178, 'page': 8, 'ar

In [13]:
# 7. Burst replay the same historical prefix with app-like headers.
source_records = load_soak_search_sequence(
    crawl_run_id=DEFAULT_SOAK_RUN_ID,
    limit=130,
)
records, path = burst_replay_request_sequence(
    source_records,
    workers=4,
    header_mode="app_like",
    label="burst-replay-soak-prefix-130-workers-4",
)
print(path)
print(make_probe_report_path(path))
print_summary(records, records_path=path)

{'replay_index': 1, 'source_request_log_id': 170, 'page': 0, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 2, 'source_request_log_id': 171, 'page': 1, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 3, 'source_request_log_id': 172, 'page': 2, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 4, 'source_request_log_id': 173, 'page': 3, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 5, 'source_request_log_id': 174, 'page': 4, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 6, 'source_request_log_id': 175, 'page': 5, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 7, 'source_request_log_id': 176, 'page': 6, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 8, 'source_request_log_id': 177, 'page': 7, 'area': '1001', 'status_code': 200, 'error_type': None}
{'replay_index': 9, 'source_request_log_id': 178, 'page': 8, 'ar

In [14]:
# 8. Re-load a saved run later and re-check the transition summary.
saved = load_jsonl(path)
print(make_probe_report_path(path))
print_summary(saved, records_path=path)

{
  "total_requests": 44,
  "status_counts": {
    "200": 43,
    "403": 1
  },
  "first_403_index": 43,
  "first_captcha_index": 43,
  "ok_before_first_403": 43,
  "ok_after_first_403": 0,
  "forbidden_after_first_403": 0,
  "first_captcha_request_id": "177421118938121aa774c5cc54f97028",
  "first_captcha_url": "https://hh.ru/account/captcha?state=pxvcxBozfu7ry7R4QCetFqDzUL2cbK2LGYV-JLOJ64744CMQTuWjk0k64Wlma3qDO6rl9aXqvWnyPUug-crFhteqVtnMEQ1bAusheRza6VE%3D",
  "first_captcha_url_with_backurl": "https://hh.ru/account/captcha?state=pxvcxBozfu7ry7R4QCetFqDzUL2cbK2LGYV-JLOJ64744CMQTuWjk0k64Wlma3qDO6rl9aXqvWnyPUug-crFhteqVtnMEQ1bAusheRza6VE%3D&backurl=http%3A%2F%2Flocalhost%3A8888%2Flab%2Ftree%2Fnotebooks%2Fhh_api_captcha_probe.ipynb"
}
{
  "last_success": {
    "timestamp_utc": "2026-03-22T20:26:29.298438+00:00",
    "params": {
      "area": "1001",
      "page": 42,
      "per_page": 20
    },
    "status_code": 200,
    "latency_ms": 429.42,
    "found": 1275,
    "pages": 64,
    "item